Final validation of required
tables, S3 exports, and latest quality run.
==

In [0]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_lakehouse"
BUCKET = "s3://ecommerce-lakehouse-1482"

required_tables = {
    "bronze": ["customers", "products", "orders", "payments"],
    "silver": [
        "customers", "products", "orders", "payments",
        "quarantine_customers", "quarantine_products",
        "quarantine_orders", "quarantine_payments"
    ],
    "gold": [
        "daily_sales", "product_performance",
        "customer_value", "dim_customer_scd2",
        "data_quality_results"
    ]
}

failures = []

In [0]:
# Verify required tables and row counts
for schema, tables in required_tables.items():
    for table in tables:
        full_name = f"{CATALOG}.{schema}.{table}"

        if not spark.catalog.tableExists(full_name):
            failures.append(f"Missing table: {full_name}")
            continue

        count = spark.table(full_name).count()
        print(f"OK | {full_name} | {count} rows")

        if count == 0 and not table.startswith("quarantine_"):
            failures.append(f"Unexpected empty table: {full_name}")

OK | ecommerce_lakehouse.bronze.customers | 15 rows
OK | ecommerce_lakehouse.bronze.products | 15 rows
OK | ecommerce_lakehouse.bronze.orders | 15 rows
OK | ecommerce_lakehouse.bronze.payments | 15 rows
OK | ecommerce_lakehouse.silver.customers | 5 rows
OK | ecommerce_lakehouse.silver.products | 5 rows
OK | ecommerce_lakehouse.silver.orders | 5 rows
OK | ecommerce_lakehouse.silver.payments | 5 rows
OK | ecommerce_lakehouse.silver.quarantine_customers | 0 rows
OK | ecommerce_lakehouse.silver.quarantine_products | 0 rows
OK | ecommerce_lakehouse.silver.quarantine_orders | 0 rows
OK | ecommerce_lakehouse.silver.quarantine_payments | 0 rows
OK | ecommerce_lakehouse.gold.daily_sales | 2 rows
OK | ecommerce_lakehouse.gold.product_performance | 3 rows
OK | ecommerce_lakehouse.gold.customer_value | 3 rows
OK | ecommerce_lakehouse.gold.dim_customer_scd2 | 9 rows
OK | ecommerce_lakehouse.gold.data_quality_results | 36 rows


In [0]:
# Verify S3 exports
for name in [
    "daily_sales",
    "product_performance",
    "customer_value",
    "dim_customer_scd2"
]:
    path = f"{BUCKET}/exports/gold/{name}"

    try:
        count = spark.read.parquet(path).count()
        print(f"OK | S3 export {name} | {count} rows")

        if count == 0:
            failures.append(f"Empty S3 export: {name}")

    except Exception as e:
        failures.append(f"S3 export failed: {name}: {str(e)[:300]}")

OK | S3 export daily_sales | 2 rows
OK | S3 export product_performance | 3 rows
OK | S3 export customer_value | 3 rows
OK | S3 export dim_customer_scd2 | 9 rows


In [0]:
# Check only the latest data-quality run
quality = spark.table(f"{CATALOG}.gold.data_quality_results")

latest_run = quality.agg(F.max("run_timestamp")).first()[0]

latest_quality = quality.filter(
    F.col("run_timestamp") == F.lit(latest_run)
)

failed_checks = latest_quality.filter(
    F.col("status") == "FAIL"
).count()

if failed_checks:
    failures.append(f"Latest data quality failures: {failed_checks}")

In [0]:
# Final validation
if failures:
    raise RuntimeError("\n".join(failures))

print("PIPELINE VALIDATION PASSED")

PIPELINE VALIDATION PASSED
